In [ ]:
!pip install -U transformers

### Downloading the **Qwen/Qwen3-0.6B** model and setting up the environment

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Used a Light Weight LLM so that the response time is fast compared to Large Models
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

### The model Implementation

In [ ]:
import re

class QwenChatbot:
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model.to("cpu")
        self.model.eval()


        self.system_prompt = {
            "role": "system",
            "content": "You are a helpful AI assistant named JARVIS. "
                       "You were created by Sayan. "
                       "Always respond in simple words and in a friendly tone."
        }

        # Initialize history with system prompt
        self.history = [self.system_prompt]

    def generate_response(self, user_input: str) -> str:
        # Add user message
        self.history.append({"role": "user", "content": user_input})

        # Limit history (Used as the Context Window for the LLM)
        MAX_HISTORY = 6
        self.history = [self.system_prompt] + self.history[-MAX_HISTORY:]

        # Build prompt
        text = self.tokenizer.apply_chat_template(
            self.history,
            tokenize=False,
            add_generation_prompt=True
        )

        # Tokenize
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            padding=True
        )

        # Generate
        output_ids = self.model.generate(
                **inputs,
                max_new_tokens=256,   # smaller = faster on CPU
                temperature=0.7,
                top_p=0.8,
                top_k=20,
                min_p=0.0,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                use_cache=True
            )

        # Extract generated tokens
        generated_ids = output_ids[0][inputs.input_ids.shape[-1]:]

        # Decode
        response = self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
        response = re.sub(r"</?think>", "", response).strip()

        # Save assistant reply
        self.history.append({"role": "assistant", "content": response})

        return response

In [ ]:
qwen_chatbot = QwenChatbot(tokenizer, model)

### The Conversation:

In [ ]:
print("👋 Hello! I am your AI assistant. How can I help you today?,\nType 'exit' to STOP CHATTING")

while True:
    user_input = input("\nUser: ")

    if user_input.strip().lower() == "exit":
        print("Bot: Have a good day!")
        break
    reply = qwen_chatbot.generate_response(user_input)

    print(f"Bot: {reply}")

👋 Hello! I am your AI assistant. How can I help you today?,
Type 'exit' to STOP CHATTING

User: Hello
Bot: Hello! How can I assist you today? 😊

User: Can you tell me something about Artificial Inetlligence, in simple words
Bot: Sure! Here's a simple explanation of Artificial Intelligence (AI) in simple terms:

**AI is a branch of computer science that helps machines think and learn.**  
Examples:  
- Self-driving cars  
- Language translation apps  
- Smart assistants like Alexa or Google Assistant  

Let me know if you'd like more examples! 😊

User: Who created Python, and what is Python
Bot: Python was created by **Guido van Rossum** in 1989. It's a programming language known for being easy to learn and widely used for various tasks like web development, data analysis, and automation. Let me know if you'd like more details! 😊

User: What is the difference between LLMs and LIMs
Bot: LLMs (Large Language Models) are AI models designed to understand and generate human-like text, such a

### Conclusion:

- I have used a **Qwen/Qwen3-0.6B** model to generate this response. The model is designed to provide accurate and relevant information based on the input it receives.
- The response is generated based on the context provided in the input, and it aims to address the user's query effectively.
- This model can work in two modes, **Thinking Mode** and **Non-Thinking Mode**, which allows it to provide more detailed and thoughtful responses when necessary. I have used the **Non-Thinking Mode** for this response to ensure clarity and conciseness.
- So that the responses I get is faster and more efficient, while still maintaining a high level of accuracy and relevance to the user's query. In-order to maintain the better user experience
